In [1]:
import torch
import torch.optim                      as optim
import numpy                            as np
import pandas                           as pd

import seaborn                          as sns
import matplotlib.pyplot                as plt
import matplotlib.gridspec              as gridspec
import plotly.express                   as px
import plotly.graph_objects             as go

from tqdm                               import tqdm
from sklearn.preprocessing              import StandardScaler, MinMaxScaler
from sklearn.cluster                    import DBSCAN, KMeans 
from sklearn.mixture                    import GaussianMixture
from transformers                       import BertTokenizer, BertForSequenceClassification
from torch.utils.data                   import DataLoader, Dataset

from utils.Preprocessing2               import PreProcessData 
from utils.plotFigures                  import plotLabels, lineAndScatterPlot, linePlot, scatterPlot2D, plot_learning_curves
from model.CustomDataset                import BertDataset, TextDataset, GPTDataset
from model.trainFunctions               import train_classification_model

# Work in Progress

In [2]:
# preData = PreProcessData()
# preData.manipulated_description = []
# for col in preData.manipulated_cr1:
#     preData.manipulated_description.append(f'{col}_description')

# df = pd.read_hdf('data\scaled_data.h5')

In [3]:
preData = PreProcessData()
df, df_healthy, scaled_data, scaled_healthy = preData.preprocessing(type_file   = 'h5', 
                                                                    scaler      = MinMaxScaler(), 
                                                                    window_size = 60)

In [4]:
df_processed = pd.read_hdf('data/df_clustering.h5')

In [5]:
cluster_model = GaussianMixture(n_components=8, n_init=5, covariance_type='tied', random_state=40)
data = df_processed[['efficiency_pacmap_x', 'efficiency_pacmap_y']].copy()
clusters = cluster_model.fit_predict(data)
data.loc[:, 'cluster'] = clusters
# scatterPlot2D(data, x='efficiency_pacmap_x', y='efficiency_pacmap_y', color='cluster')

In [6]:
df_processed.loc[:, 'efficiency_cluster_pacmap'] = clusters
df_processed.loc[(df_processed['efficiency_pacmap_y']>-1.7)&(df_processed['efficiency_pacmap_x']<8.7)&(df_processed['efficiency_cluster_pacmap']==4), 'efficiency_cluster_pacmap']=0
df_processed.loc[(df_processed['efficiency_pacmap_y']>1.7)&(df_processed['efficiency_pacmap_x']<8.75)&(df_processed['efficiency_cluster_pacmap']==4), 'efficiency_cluster_pacmap']=0
df_processed.loc[:, 'efficiency_cluster_pacmap']= df_processed['efficiency_cluster_pacmap'].replace({5:1, 6:1, 4:3, 7:0 })

C:\Users\CAEX\AppData\Local\Temp\ipykernel_3912\2023452656.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_processed.loc[:, 'efficiency_cluster_pacmap']= df_processed['efficiency_cluster_pacmap'].replace({5:1, 6:1, 4:3, 7:0 })


In [7]:
scatterPlot2D(df_processed, x='efficiency_pacmap_x', y='efficiency_pacmap_y', color='efficiency_cluster_pacmap')

In [8]:
data        = df_processed.copy()
color       = 'efficiency_cluster_pacmap'
data[color] = data[color].astype('str')
fig = px.scatter(data, 
                    x      = 'tiempo', 
                    y      = preData.efficiency_cr1[0], 
                    color  = color)
fig.update_traces(line=dict(color='white'))

fig.update_layout(template='plotly_dark',
                  plot_bgcolor='rgba(0, 0, 0, 0)',
                  paper_bgcolor='rgba(0, 0, 0, 0)',
                  xaxis=dict(showgrid=False),
                  yaxis=dict(showgrid=False),
                  height=300
                  )
fig.show()

In [9]:
# df_avg = df_processed.select_dtypes('number').rolling(window=10, min_periods=1).mean()
# df_avg.loc[:, 'target'] = df_processed['efficiency_cluster_pacmap'].rolling(window=10, min_periods=1).apply(lambda x: x.mode().iloc[0], raw=False)
# df_avg.loc[:, 'text']   = df_processed['manipulable_cluster_pacmap'].rolling(window=10, min_periods=1).apply(lambda x: x.mode().iloc[0], raw=False)

In [10]:
df_aggregation = df_processed.set_index('tiempo').select_dtypes('number').resample('10min').mean()
df_aggregation.loc[:, 'efficiency_cluster_pacmap'] = df_processed.set_index('tiempo')['efficiency_cluster_pacmap'].resample('10min').apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
df_aggregation.loc[:, 'manipulable_cluster_pacmap'] = df_processed.set_index('tiempo')['manipulable_cluster_pacmap'].resample('10min').apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
df_aggregation.loc[:, 'trajectory'] = df_processed.set_index('tiempo')['trajectory'].resample('10min').apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
df_aggregation.reset_index(inplace=True)
df_aggregation.dropna(inplace=True)

In [11]:
df_aggregation.loc[:, 'text'] = df_aggregation['manipulable_cluster_pacmap'].replace({0: 'A', 1: 'B', 2:'C'})
df_aggregation.rename(columns={'efficiency_cluster_pacmap': 'target'}, inplace=True)

In [12]:
# excluded_columns = ['Eje_piñón_aceite(%)_description',          'Lubricación_respaldo(psi)_description', 
#                     'Lubricación_respaldo(A)_description',      'Motor_Fase2(A)_description', 
#                     'Motor_Fase3(A)_description',               'Motor_Fase2(V)_description', 
#                     'Motor_Fase3(V)_description']

In [13]:
# df.loc[:, 'text'] = df[preData.manipulated_description].drop(columns=excluded_columns).filter(like='description').apply(lambda row: "[CLS] " + " [SEP] ".join(row.to_list()) + " [SEP]", axis=1)

# Experiment 2

## Holdout

In [14]:
val_trajectories = ['15', '29', '63']
test_trajectories = ['9', '17', '51']

train_data  = df_processed[~df_processed['trajectory'].isin(test_trajectories+val_trajectories)].sample(frac=1)
val_data    = df_processed[df_processed['trajectory'].isin(val_trajectories)].sample(frac=1)
test_data   = df_processed[df_processed['trajectory'].isin(test_trajectories)].sample(frac=1)

## Dataloader

In [15]:
# Parameters
MAX_LEN = 512
BATCH_SIZE = 32
model_name  = "huawei-noah/TinyBERT_General_4L_312D" #'bert-base-uncased'
tokenizer   = BertTokenizer.from_pretrained(model_name)

train_dataset = BertDataset(
    texts       = train_data['text'].tolist(),
    labels      = train_data['efficiency_cluster_pacmap'].tolist(),
    tokenizer   = tokenizer,
    max_len     = MAX_LEN
)

val_dataset = BertDataset(
    texts       = val_data['text'].tolist(),
    labels      = val_data['efficiency_cluster_pacmap'].tolist(),
    tokenizer   = tokenizer,
    max_len     = MAX_LEN
)

test_dataset = BertDataset(
    texts       = test_data['text'].tolist(),
    labels      = test_data['efficiency_cluster_pacmap'].tolist(),
    tokenizer   = tokenizer,
    max_len     = MAX_LEN
)

train_loader    = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader      = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader     = DataLoader(test_dataset,  batch_size=1)

KeyError: 'text'

# Experiment 1

In [ ]:
val_trajectories = ['15', '29', '63']
test_trajectories = ['9', '17', '51']

train_data  = df_aggregation[~df_aggregation['trajectory'].isin(test_trajectories+val_trajectories)]#.sample(frac=1)
val_data    = df_aggregation[df_aggregation['trajectory'].isin(val_trajectories)]#.sample(frac=1)
test_data   = df_aggregation[df_aggregation['trajectory'].isin(test_trajectories)]#.sample(frac=1)

In [ ]:
MAX_LEN     = 112
BATCH_SIZE  = 256
WINDOW_SIZE = 100
model_name  = "huawei-noah/TinyBERT_General_4L_312D" #'bert-base-uncased'
tokenizer   = BertTokenizer.from_pretrained(model_name)
#tokenizer.pad_token = tokenizer.eos_token #GPT2

train_dataset = TextDataset(
    texts       = train_data[['text']],
    labels      = train_data['target'].tolist(),
    tokenizer   = tokenizer,
    window_size = WINDOW_SIZE, 
    max_len     = MAX_LEN
)

val_dataset = TextDataset(
    texts       = val_data[['text']],
    labels      = val_data['target'].tolist(),
    tokenizer   = tokenizer,
    window_size = WINDOW_SIZE, 
    max_len     = MAX_LEN
)

test_dataset = TextDataset(
    texts       = test_data[['text']],
    labels      = test_data['target'].tolist(),
    tokenizer   = tokenizer,
    window_size = WINDOW_SIZE, 
    max_len     = MAX_LEN
)

train_loader    = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader      = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader     = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

In [ ]:
data        = df_aggregation.copy()
color       = 'target'
data[color] = data[color].astype('str')
data['line'] = 0
fig = px.scatter(data, 
                    x      = 'tiempo', 
                    y      = 'line',#preData.efficiency_cr1[0], 
                    color  = color)
fig.update_traces(line=dict(color='white'))

fig.update_layout(template='plotly_dark',
                  plot_bgcolor='rgba(0, 0, 0, 0)',
                  paper_bgcolor='rgba(0, 0, 0, 0)',
                  xaxis=dict(showgrid=False),
                  yaxis=dict(showgrid=False),
                  height=300
                  )
fig.show()

# Training Model

In [ ]:
model       = BertForSequenceClassification.from_pretrained(model_name, num_labels=len(train_data.target.unique()))

#Freezing
for param in model.bert.parameters():
    param.requires_grad = False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
optimizer   = optim.AdamW(model.parameters(), lr=0.001)
EPOCHS      = 1

train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve = train_classification_model(model             = model, 
                                                                                              optimizer         = optimizer, 
                                                                                              train_iterator    = train_loader, 
                                                                                              valid_iterator    = val_loader,
                                                                                              test_iterator     = test_loader, 
                                                                                              epochs            = EPOCHS,
                                                                                              dir_model         = 'saved_models/tiny_bert_02_07.pt')
# Learning curves
# plot_learning_curves(train_acc_curve, val_acc_curve, train_loss_curve, val_loss_curve)

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

device = 'cuda'
# Cambia el modelo a modo de evaluación
model.eval()

# Lista para almacenar las predicciones y etiquetas reales
all_predictions = []
all_labels = []

# Desactiva el cálculo de gradientes
with torch.no_grad():
    for batch in test_loader:
        input_ids       = batch['input_ids'].to(device)
        attention_mask  = batch['attention_mask'].to(device)
        labels          = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask).logits
        
        # Aplica softmax para convertir los logits en probabilidades
        probabilities = torch.softmax(outputs, dim=1)
        
        # Obtén las predicciones (la clase con mayor probabilidad)
        _, predicted = torch.max(probabilities, 1)
        
        # Almacena las predicciones y las etiquetas reales
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calcula la matriz de confusión
cm = confusion_matrix(all_labels, all_predictions)

# Configuración para la visualización
plt.figure(figsize=(10, 7))
sns.set(font_scale=1.4)  # Para aumentar el tamaño de la fuente en la gráfica

# Graficar la matriz de confusión
sns.heatmap(cm, annot=True, annot_kws={"size": 16}, cmap='Blues', fmt='g')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')

# Mostrar la gráfica
plt.show()


In [ ]:
lineAndScatterPlot(df, x='tiempo', y=preData.manipulated_cr1[6], color=preData.manipulated_description[6])